# 機率/統計之機器學習基礎應用

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 區分離散型與連續型隨機變數，並理解 PMF 與 PDF 的差異。
2. 使用 Python 模擬伯努利、二項、泊松與常態分佈。
3. 以條件機率與貝氏定理更新事件發生機率。
4. 使用假設檢定與 p 值輔助模型或特徵判斷。
5. 將機率概念連結到機器學習分類模型的預測機率。

本練習聚焦於中級 AI 應用規劃師常見的實務理解：模型不是只輸出答案，而是根據資料估計「某結果發生的可能性」。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並設定隨機種子與圖表樣式，方便後續模擬結果可重現。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('環境設定完成')
print('numpy 版本:', np.__version__)
print('pandas 版本:', pd.__version__)


## 核心概念說明

在機器學習中，我們通常不是直接尋找唯一且絕對的答案，而是估計條件機率：

$$P(Y|X)$$

其中，`X` 代表輸入特徵，`Y` 代表目標變數。分類模型輸出的「類別機率」就是這個概念的實務形式。

### 離散型隨機變數

離散型隨機變數的可能值有限或可數，例如：是否點擊、是否通過審核、一天內客服來電次數。常見分佈包含：

- 伯努利分佈：一次成功或失敗事件。
- 二項分佈：多次伯努利事件中的成功次數。
- 泊松分佈：固定時間或空間內事件發生次數。

### 連續型隨機變數

連續型隨機變數可取任意實數，例如：等待時間、溫度、分數、身高。常見分佈包含：

- 常態分佈：常用於誤差、量測值與標準化特徵。
- 均勻分佈：每個區間內數值出現機率相同。
- 指數分佈：常用於等待時間或事件間隔時間。


In [ ]:
# ── 示範：離散型機率分佈 ──────────────────────────────
# 這段程式碼示範伯努利、二項與泊松分佈，並用模擬結果理解不同分佈適合描述的資料型態。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# 伯努利分佈：一次事件是否成功，例如使用者是否點擊廣告
p_click = 0.3
bernoulli_samples = stats.bernoulli.rvs(p_click, size=1000)

# 二項分佈：20 次曝光中，使用者點擊幾次
binomial_samples = stats.binom.rvs(n=20, p=p_click, size=1000)

# 泊松分佈：每小時平均 4 通客服來電
poisson_samples = stats.poisson.rvs(mu=4, size=1000)

summary = pd.DataFrame({
    '分佈': ['伯努利', '二項', '泊松'],
    '模擬情境': ['單次是否點擊', '20 次曝光中的點擊數', '每小時客服來電數'],
    '樣本平均': [bernoulli_samples.mean(), binomial_samples.mean(), poisson_samples.mean()],
    '樣本變異數': [bernoulli_samples.var(), binomial_samples.var(), poisson_samples.var()]
})

print(summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(bernoulli_samples, bins=[-0.5, 0.5, 1.5], rwidth=0.6)
axes[0].set_title('伯努利分佈：是否點擊')
axes[0].set_xticks([0, 1])

axes[1].hist(binomial_samples, bins=range(0, 22), rwidth=0.8)
axes[1].set_title('二項分佈：20 次中的成功次數')

axes[2].hist(poisson_samples, bins=range(0, 13), rwidth=0.8)
axes[2].set_title('泊松分佈：每小時事件數')

for ax in axes:
    ax.set_xlabel('取值')
    ax.set_ylabel('出現次數')

plt.tight_layout()
plt.show()


## 條件機率與貝氏推論

條件機率描述的是：在某條件已知的前提下，另一事件發生的機率。

$$P(A|B)=\frac{P(A \cap B)}{P(B)}$$

貝氏定理則進一步說明如何用新觀察更新原本的信念：

$$P(A|B)=\frac{P(B|A)P(A)}{P(B)}$$

在機器學習中，這可對應到：

- 先驗機率：尚未觀察新資料前的初始信念。
- 似然：假設某狀態成立時，觀察到目前資料的可能性。
- 後驗機率：整合新資料後更新的機率。

例如在風險預測、醫療診斷、垃圾郵件分類或推薦系統中，模型會根據新觀察到的特徵，更新某個結果發生的可能性。


In [ ]:
# ── 示範：貝氏定理更新風險機率 ───────────────────────────
# 這段程式碼用簡化的醫療檢測情境示範貝氏定理：即使檢測準確率很高，若疾病本身很罕見，陽性後的實際罹病機率仍需透過條件機率重新計算。

import numpy as np

# 情境：某疾病盛行率為 1%，檢測靈敏度 95%，偽陽性率 5%
prior_disease = 0.01              # P(疾病)
sensitivity = 0.95                # P(陽性 | 疾病)
false_positive_rate = 0.05        # P(陽性 | 無疾病)
prior_no_disease = 1 - prior_disease

# P(陽性) = P(陽性|疾病)P(疾病) + P(陽性|無疾病)P(無疾病)
p_positive = sensitivity * prior_disease + false_positive_rate * prior_no_disease

# P(疾病|陽性) = P(陽性|疾病)P(疾病) / P(陽性)
posterior_disease_given_positive = sensitivity * prior_disease / p_positive

print(f'疾病先驗機率 P(疾病): {prior_disease:.2%}')
print(f'檢測陽性的總機率 P(陽性): {p_positive:.2%}')
print(f'陽性後的後驗機率 P(疾病|陽性): {posterior_disease_given_positive:.2%}')

if posterior_disease_given_positive < 0.5:
    print('解讀：檢測陽性會提高風險估計，但仍不代表一定罹病，需結合更多資訊。')
else:
    print('解讀：檢測陽性後風險已明顯升高，需進一步確認。')


## 從統計推論連到機器學習模型

統計推論常用來判斷觀察到的差異是否可能只是隨機變異造成。例如在特徵選擇或 A/B 測試中，我們會使用假設檢定與 p 值來輔助判斷。

在分類模型中，機器學習模型通常會輸出每個類別的預測機率。以邏輯迴歸為例，它適合用於二元分類任務，可被視為估計：

$$P(Y=1|X)$$

因此，機率與統計不是額外的理論，而是直接影響模型假設、模型輸出解讀與決策門檻設定的基礎。


In [ ]:
# ── 實際應用：分類模型的預測機率與假設檢定 ─────────────────────
# 這段程式碼建立一個二元分類資料集，訓練邏輯迴歸模型，觀察模型輸出的預測機率，並使用 t 檢定比較某個特徵在兩類之間是否有顯著差異。

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

np.random.seed(42)

X, y = make_classification(
    n_samples=600,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    class_sep=1.2,
    random_state=42
)

feature_names = ['特徵1', '特徵2', '特徵3', '特徵4']
df = pd.DataFrame(X, columns=feature_names)
df['目標'] = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

pred_label = model.predict(X_test)
pred_prob = model.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, pred_label):.3f}')
print(f'Log loss: {log_loss(y_test, pred_prob):.3f}')

result = pd.DataFrame({
    '實際類別': y_test[:10],
    '預測為類別1的機率': np.round(pred_prob[:10], 3),
    '預測類別': pred_label[:10]
})
print('\n前 10 筆預測結果：')
print(result)

# 假設檢定：比較特徵1在目標類別 0 與 1 之間的平均數是否不同
group_0 = df[df['目標'] == 0]['特徵1']
group_1 = df[df['目標'] == 1]['特徵1']
t_stat, p_value = stats.ttest_ind(group_0, group_1, equal_var=False)

print('\n特徵1的兩組平均數比較：')
print(f'類別 0 平均: {group_0.mean():.3f}')
print(f'類別 1 平均: {group_1.mean():.3f}')
print(f't statistic: {t_stat:.3f}')
print(f'p-value: {p_value:.4f}')

alpha = 0.05
if p_value < alpha:
    print('解讀：p 值小於 0.05，表示特徵1在兩個類別之間的平均數差異具有統計顯著性。')
else:
    print('解讀：p 值不小於 0.05，尚無足夠證據認為特徵1在兩個類別之間的平均數不同。')
